In [1]:
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import math
import tqdm
from datetime import datetime

In [2]:
ukbiobank = pd.read_csv('./NAFLD-CVD/data/ukb_first.csv', low_memory=False, chunksize=100000)
chunk=list(ukbiobank)

In [3]:
columnlist = pd.read_csv('./NAFLD-CVD/data/column_list.csv')

In [5]:
columnlist

,Field ID,Description,Category
0,21022,Age at recruitment,Baseline characteristics
1,31,Sex,Baseline characteristics
2,41270,Diagnoses - ICD10,Summary Diagnoses
3,41202,Diagnoses - main ICD10,Summary Diagnoses
4,41204,Diagnoses - secondary ICD10,Summary Diagnoses
...,...,...,...
68,41280,Date of first in-patient diagnosis - ICD10,Summary Diagnoses
69,41281,Date of first in-patient diagnosis - ICD9,Summary Diagnoses
70,41262,Date of first in-patient diagnosis - main ICD10,Summary Diagnoses
71,41263,Date of first in-patient diagnosis - main ICD9,Summary Diagnoses


In [4]:
split1 = chunk[0].copy()
split2 = chunk[1].copy()
split3 = chunk[2].copy()
split4 = chunk[3].copy()
split5 = chunk[4].copy()
split6 = chunk[5].copy()

In [5]:
split_l = [split1, split2, split3, split4, split5, split6]

### Age

In [6]:
# remove outlier

tmp_l = []
for split_df in split_l:
    colname = '21022-0.0'
    tmp = split_df.query('40 <= `{}` <= 69'.format(colname))
    tmp_l.append(tmp)

In [7]:
combined_df = pd.concat([tmp_l[0], tmp_l[1], tmp_l[2], tmp_l[3], tmp_l[4], tmp_l[5]])

In [9]:
len(combined_df)

499963

# Exclusion

### 1. Alcohol intake

In [7]:
drink_more_than_20g = pd.read_csv('./NAFLD-CVD/data/drink_more_than_20g.csv')
drink_more_than_20g_eid = drink_more_than_20g.eid.tolist()

In [8]:
drink_more_than_20g

,eid,1568-0.0,1568-1.0,1568-2.0,1568-3.0,1578-0.0,1578-1.0,1578-2.0,1578-3.0,1588-0.0,...,20013-3.25,20013-3.26,20013-3.27,20013-3.28,20013-3.29,20013-3.30,20013-3.31,20013-3.32,20013-3.33,alcohol_consume2
0,1000119,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.000000
1,1000160,6.0,NaN,NaN,NaN,6.0,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.057143
2,1000187,15.0,NaN,NaN,NaN,6.0,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.400000
3,1000432,9.0,NaN,11.0,NaN,12.0,NaN,11.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.400000
4,1000476,3.0,NaN,NaN,NaN,6.0,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55546,6023670,0.0,NaN,NaN,NaN,6.0,NaN,NaN,NaN,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.542857
55547,6023688,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.857143
55548,6023865,2.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.457143
55549,6023882,18.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48.457143


In [11]:
man_or_woman = combined_df.query('eid in @drink_more_than_20g_eid')

In [12]:
man_or_woman['31-0.0'] # female : 0 , male : 1
man_or_woman = man_or_woman.merge(drink_more_than_20g[['eid','alcohol_consume2']], on=['eid'])
man_or_woman = man_or_woman[(man_or_woman['31-0.0'] == 0) & (man_or_woman['alcohol_consume2'] >= 20) 
                            | (man_or_woman['31-0.0'] == 1) & (man_or_woman['alcohol_consume2'] >= 30)]

man_or_woman_eid = man_or_woman.eid.tolist()

In [13]:
combined_df = combined_df.query('eid not in @man_or_woman_eid')

In [16]:
len(combined_df)

456436

### 2. Exclude FLI cannot be calculated

In [11]:
data3 = pd.read_csv('./NAFLD-CVD/data/data3.csv', usecols=['eid','53-0.0'])
combined_df = combined_df.merge(data3, on=['eid'])
data4 = pd.read_csv('./NAFLD-CVD/data/data4.csv', usecols=['eid','30080-0.0', '30870-0.0'])
combined_df = combined_df.merge(data4, on=['eid'])

In [19]:
FLI_calc_available = combined_df.copy()

In [20]:
FLI_calc_available = FLI_calc_available[['eid','30870-0.0','48-0.0','30730-0.0','21001-0.0']]

In [21]:
#missing이 있는 경우 제거
FLI_calc_available.dropna(axis = 0, how = 'any', inplace = True)

In [22]:
# 30870 : Total Triglycerides
# 48 : Waist circumference
# 30730 : Gamma glutamyltransferase
# 21001, 23104 : BMI

def fli_calculator(row):
    numerator = (math.exp(0.953 * math.log(row['30870-0.0']*18) + 0.139 * row['21001-0.0'] +
                          0.718 * math.log(row['30730-0.0']) + 0.053 * row['48-0.0'] - 15.745))
    denominator = (1 + (math.exp(0.953 * math.log(row['30870-0.0']*18) + 0.139 * row['21001-0.0'] +
                          0.718 * math.log(row['30730-0.0']) + 0.053 * row['48-0.0'] - 15.745)))
    result = (numerator / denominator) * 100
    return result


In [23]:
FLI_calc_available['FLI'] = FLI_calc_available.apply(fli_calculator, axis=1)

In [24]:
combined_df = combined_df.merge(FLI_calc_available, on='eid')

### 3. CVD History / Events before recruit

In [28]:
data = combined_df.copy()
del combined_df
import tqdm
import numpy as np

# List of ICD-10 codes for diseases to be filtered out

'''
Stroke
I60, I61, I63, I64

Heart failure
I50

MI
I21-23, I241, I252

'''
icd_10_events_codes = ['I20', 'I25', 'I21', 'I22', 'I23', 'I24', 'I25', 'I50', 'I60', 'I61', 'I63', 'I64']
icd_10_liver_disease_codes = ['K70', 'K71', 'B16', 'B17', 'B18', 'B19']
icd_10_other_disease = ['K743', 'K754', 'K830' 'E831', 'E830', 'E880', 'I820', 'K765', 'K744', 'K745']

# Recruitment date column
recruitment_date_column = '53-0.0'

# Identify columns starting with '41202-' and '41262-'
icd10_columns = [col for col in data.columns if col.startswith('41202-')]
icd10_date_columns = [col for col in data.columns if col.startswith('41262-')]

# Convert recruitment date to datetime
data[recruitment_date_column] = pd.to_datetime(data[recruitment_date_column])

events_codes_set = set(icd_10_events_codes)
liver_disease_codes_set = set(icd_10_liver_disease_codes)
other_disease_codes_set = set(icd_10_other_disease)


def should_drop_cvd(row):
    for icd_col, date_col in (zip(icd10_columns, icd10_date_columns)):
        # Convert diagnosis date to datetime
        diagnosis_date = pd.to_datetime(row[date_col])

        # If the row is NaN, skip this iteration
        if pd.isna(row[icd_col]):
            continue

        # Check if patient had stroke or AMI before recruitment
        if any(row[icd_col].startswith(code) for code in events_codes_set):
            if diagnosis_date < row[recruitment_date_column]:
                return True
    # If none of the conditions were met, do not drop the row
    return False

def should_drop_liver(row):
    for icd_col, date_col in (zip(icd10_columns, icd10_date_columns)):
        # Convert diagnosis date to datetime
        diagnosis_date = pd.to_datetime(row[date_col])

        # If the row is NaN, skip this iteration
        if pd.isna(row[icd_col]):
            continue

        # Check if patient was diagnosed with liver disease or hepatitis after recruitment
        if any(row[icd_col].startswith(code) for code in liver_disease_codes_set):
            if diagnosis_date > row[recruitment_date_column]:
                return True

        elif row[icd_col] in other_disease_codes_set:
            if diagnosis_date > row[recruitment_date_column]:
                return True

    # If none of the conditions were met, do not drop the row
    return False

In [29]:
to_drop = data.apply(should_drop_cvd, axis=1)
data['icd10_exclude_disease_cvd'] = to_drop.astype(int)
to_drop = data.apply(should_drop_liver, axis=1)
data['icd10_exclude_disease_liver'] = to_drop.astype(int)

In [30]:
data.icd10_exclude_disease_cvd.value_counts()

icd10_exclude_disease_cvd
0    408297
1     15829
Name: count, dtype: int64

In [31]:
data = data.query("icd10_exclude_disease_cvd == 0")

In [32]:
len(data)

408297

In [34]:
data.icd10_exclude_disease_liver.value_counts()

icd10_exclude_disease_liver
0    407453
1       844
Name: count, dtype: int64

In [35]:
data = data.query("icd10_exclude_disease_liver == 0")

In [36]:
len(data)

407453

### condition 4 - FIB-4, ASCVD Risk, Framingham risk, SAFE score cannot calculated

#### FIB-4

In [51]:
def fib4_calculator(row):
    numerator = (row['21022-0.0'] * row['30650-0.0'])
    denominator = (row['30080-0.0'] * math.sqrt(row['30620-0.0']))
    result = (numerator / denominator)
    return result


data['FIB-4'] = data.apply(fib4_calculator, axis=1)

#### ASCVD Risk

##### DM, BP treated definition needed

In [52]:
from datetime import datetime
import tqdm

icd_10_dm = ['E10', 'E11', 'E12', 'E13', 'E14']  #startswith

data['DM_icd10'] = 0

for i in tqdm.tqdm(range(data.shape[0])):
    for col in data.columns:
        # Check for columns starting with '41202-' for ICD10 codes
        if str(col).startswith('41202'):
            # Get the corresponding date column
            date_col = '41262' + col[5:]
            # Check if the date column exists in the dataframe
            try:
                input_str = (data.at[i, '53-0.0'])
                datetime_obj = datetime.strptime(input_str, "%Y-%m-%d")
                input_str2 = str(data.at[i, date_col])
                datetime_obj2 = datetime.strptime(input_str2, "%Y-%m-%d")
                if (datetime_obj2 < datetime_obj) and any(str(data.at[i, col]).startswith(code) for code in icd_10_dm):
                    data.at[i, 'DM_icd10'] = 1

                    break
            except Exception as e:
                continue

100%|█████████████████████████████████| 407453/407453 [05:29<00:00, 1236.98it/s]


In [56]:
blood_chem = pd.read_csv('./NAFLD-CVD/data/data6.txt', sep='\t', usecols=range(1, 61))
blood_chem.index = blood_chem.index.droplevel()
blood_chem = blood_chem.reset_index()

In [57]:
blood_chem = blood_chem[['index', '30600-0.0', '30610-0.0', '30620-0.0',
                         '30630-0.0', '30640-0.0', '30650-0.0', '30660-0.0', '30670-0.0', '30680-0.0', '30690-0.0',
                         '30700-0.0', '30710-0.0', '30720-0.0', '30730-0.0', '30740-0.0', '30750-0.0', '30760-0.0',
                         '30770-0.0',
                         '30780-0.0', '30790-0.0', '30800-0.0', '30810-0.0', '30820-0.0', '30830-0.0', '30840-0.0',
                         '30850-0.0', '30860-0.0', '30870-0.0', '30880-0.0', '30890-0.0'
                         ]]
blood_chem.columns = ['eid', 'Albumin', 'Alkaline_phosphatase', 'Alanine_aminotransferase', 'Apolipoprotein_A',
                      'Apolipoprotein_B',
                      'Aspartate_aminotransferase', 'Direct bilirubin', 'Urea', 'Calcium', 'Cholesterol', 'Creatinine',
                      'C-reactive_protein',
                      'Cystatin_C', 'Gamma_glutamyltransferase', 'Glucose', 'HbA1c', 'HDL', 'IGF-1', 'LDL_direct',
                      'Lipoprotein_A',
                      'Oestradiol', 'Phosphate', 'Rheumatoid_factor', 'SHBG', 'Total_bilirubin', 'Testosterone',
                      'Total_protein', 'Triglycerides',
                      'Urate', 'Vitamin_D']
data = data.merge(blood_chem, on=['eid'])

In [58]:
#### glucose
data['DM_glucose'] = (data['Glucose'] * 18.018 >= 200).astype(int)
#### HbA1c
data['DM_HbA1c'] = ((data['HbA1c'] / 10.929) + 2.15 >= 6.5).astype(int)

In [59]:

#### DM medication

# List of keywords associated with diabetes medications and insulin
diabetes_keywords = [
    "metformin", "sulfonylurea", "thiazolidinedione",
    "dpp-4 inhibitor", "sglt2 inhibitor", "insulin",
    "glipizide", "glyburide", "glimepiride", "pioglitazone",
    "rosiglitazone", "sitagliptin", "linagliptin", "empagliflozin",
    "dapagliflozin", "canagliflozin"
]

medication = pd.read_csv('./NAFLD-CVD/data/data5.csv')
# Load the medication code list from the provided file
medication_code = pd.read_csv('./NAFLD-CVD/data/coding4.tsv', delimiter='\t',
                              encoding='utf-8')

# Filter the medication dataframe based on the keywords
diabetes_meds = medication_code[medication_code['meaning'].str.lower().str.contains('|'.join(diabetes_keywords))]

dm_med_list = diabetes_meds.coding.tolist()
# Identify columns that start with '20003-0.' and change their dtype to string
cols_to_convert = [col for col in medication.columns if col.startswith('20003-0.')]

medication[cols_to_convert] = medication[cols_to_convert].astype(str)
# Identify columns that start with '20003' 
cols_to_modify = [col for col in medication.columns if col.startswith('20003')]

# For each column, split the value at the dot and keep the portion before the dot
for col in cols_to_modify:
    try:
        medication[col] = medication[col].str.split('.').str[0]
    except:
        continue

# Display the first few rows of the modified dataframe to verify changes
medication[cols_to_modify].head()
cols_to_modify = [col for col in medication.columns if col.startswith('20003-0.')]

# Given the previously identified diabetes medication codes as dm_med_list
dm_med_list = ['1140874646', '1140883066', '1140884600', '1141152590',
               '1141157284', '1141171646', '1141177600', '1141189090']

# Create a new column 'DM_medication' and initialize with 0
medication['DM_medication'] = 0

# For each column starting with '20003-', if its value is in dm_med_list, update 'DM_medication' column to 1
for col in cols_to_modify:
    medication.loc[medication[col].isin(dm_med_list), 'DM_medication'] = 1

# Display the first few rows to verify the changes
medication[['DM_medication'] + cols_to_modify].head()
data = data.merge(medication[['eid', 'DM_medication']], on=['eid'])

/var/folders/9c/cy0gwyx13172k5yh7sbsrw9r0000gn/T/ipykernel_77054/2620655772.py:12: DtypeWarning: Columns (221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,243,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276) have mixed types. Specify dtype option on import or set low_memory=False.
  medication = pd.read_csv('/Users/charmbong/Documents/NAFLD-CVD/data/data5.csv')


In [60]:
bp_treated_man = medication[medication['6177-0.0'] == 2.0]
bp_treated_woman = medication[medication['6153-0.0'] == 2.0]
bp_l_m = bp_treated_man.eid.tolist()
bp_l_wm = bp_treated_woman.eid.tolist()
dm_treated_man = medication[medication['6177-0.0'] == 3.0]
dm_treated_woman = medication[medication['6153-0.0'] == 3.0]
dm_l_m = dm_treated_man.eid.tolist()
dm_l_wm = dm_treated_woman.eid.tolist()
data['BP_treated'] = data['eid'].apply(lambda x: 1 if x in bp_l_m or x in bp_l_wm else 0)
data['insulin'] = data['eid'].apply(lambda x: 1 if x in dm_l_m or x in dm_l_wm else 0)

In [61]:
#### Fianal DM
data['DM'] = (data['DM_icd10'] | data['DM_medication'] | data['insulin'] | data['DM_glucose'] | data['DM_HbA1c']).astype(int)

In [64]:
ethnics = pd.read_csv(
    "./NAFLD-CVD/data/Socio-demographics.csv"
    , usecols=['eid', '21000-0.0'])
data = data.merge(ethnics, on=['eid'])
del ethnics
del blood_chem
del medication
del medication_code

- Sex : '31-0.0'
- ethinics : '21000-0.0'
- Smoke status : 1239-0.0 / 20116-0.0
- BP treated : 
- DM :
- Age : 21022-0.0
- Systolic : 4080-0.0
- HDL : '30760-0.0'
- LDL : '30780-0.0'
- Triglycerides : 30870-0.0

In [65]:
# Categorical data modified
data['20116-0.0'].loc[data['20116-0.0'] == -3] = 0

/var/folders/9c/cy0gwyx13172k5yh7sbsrw9r0000gn/T/ipykernel_77054/2989603723.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['20116-0.0'].loc[data['20116-0.0'] == -3] = 0


In [66]:
def compute_ten_year_score(df, sex, ethnics, smoke, bp_treated, dm_treated, age_col, sbp, hdl_col, ldl_col, trigly_col):
    """
    Args:
        isMale (bool)
        isBlack (bool)
        smoker (bool)
        hypertensive (bool)
        diabetic (bool)
        age (int)
        systolicBloodPressure (int)
        totalCholesterol (int)
        hdl (int)
    """
    def ascvd(row):
        if row[sex] == 1:
            isMale = True
        else:
            isMale = False
        if str(row[ethnics]).split('.')[0].startswith('4'):
            isBlack = True
        else:
            isBlack = False
        if row[smoke] > 1:
            smoker = True
        else:
            smoker = False
        if row[bp_treated] == 1:
            hypertensive = True
        else:
            hypertensive = False
        if row[dm_treated] == 1:
            diabetic = True
        else:
            diabetic = False
        age = row[age_col]
        hdl = row[hdl_col] * 38.67
        ldl = row[ldl_col] * 38.67
        trigly = row[trigly_col] * 88.57
        systolicBloodPressure = row[sbp]
        totalCholesterol = hdl + ldl + (trigly * 0.2)
        treatment_for_hypertension = row['BP_treated']
        
        if age < 40 or age > 79:
            return None
        lnAge = math.log(age)
        lnTotalChol = math.log(totalCholesterol)
        lnHdl = math.log(hdl)
        trlnsbp = math.log(systolicBloodPressure) if hypertensive else 0
        ntlnsbp = 0 if hypertensive else math.log(systolicBloodPressure)
        ageTotalChol = lnAge * lnTotalChol
        ageHdl = lnAge * lnHdl
        agetSbp = lnAge * trlnsbp
        agentSbp = lnAge * ntlnsbp
        ageSmoke = lnAge if smoker else 0
        if isBlack and not isMale:
            s010Ret = 0.95334
            mnxbRet = 86.6081
            predictRet = (
                17.1141 * lnAge
                + 0.9396 * lnTotalChol
                + -18.9196 * lnHdl
                + 4.4748 * ageHdl
                + 29.2907 * trlnsbp
                + -6.4321 * agetSbp
                + 27.8197 * ntlnsbp
                + -6.0873 * agentSbp
                + (0.6908 if smoker else 0)
                + (0.8738 if diabetic else 0)
            )
        elif not isBlack and not isMale:
            s010Ret = 0.96652
            mnxbRet = -29.1817
            predictRet = (
                -29.799 * lnAge
                + 4.884 * lnAge ** 2
                + 13.54 * lnTotalChol
                + -3.114 * ageTotalChol
                + -13.578 * lnHdl
                + 3.149 * ageHdl
                + 2.019 * trlnsbp
                + 1.957 * ntlnsbp
                + (7.574 if smoker else 0)
                + -1.665 * ageSmoke
                + (0.661 if diabetic else 0)
            )
        elif isBlack and isMale:
            s010Ret = 0.89536
            mnxbRet = 19.5425
            predictRet = (
                2.469 * lnAge
                + 0.302 * lnTotalChol
                + -0.307 * lnHdl
                + 1.916 * trlnsbp
                + 1.809 * ntlnsbp
                + (0.549 if smoker else 0)
                + (0.645 if diabetic else 0)
            )
        else:
            s010Ret = 0.91436
            mnxbRet = 61.1816
            predictRet = (
                12.344 * lnAge
                + 11.853 * lnTotalChol
                + -2.664 * ageTotalChol
                + -7.99 * lnHdl
                + 1.769 * ageHdl
                + 1.797 * trlnsbp
                + 1.764 * ntlnsbp
                + (7.837 if smoker else 0)
                + -1.795 * ageSmoke
                + (0.658 if diabetic else 0)
            )

        pct = 1 - s010Ret ** math.exp(predictRet - mnxbRet)
        try:
            return round(pct * 100 * 10) / 10
        except:
            return np.nan
    df['ASCVD_Risk'] = df.apply(ascvd, axis=1)
    return df

In [67]:
# compute_ten_year_score(df, sex, ethnics, smoke, bp_treated, dm_treated, age, sbp, hdl, ldl, trigly)
data = compute_ten_year_score(data, '31-0.0', '21000-0.0', '20116-0.0', 'BP_treated',
                                         'DM', '21022-0.0', '4080-0.0', 'HDL', 'LDL_direct', 'Triglycerides'
                                         )

#### FRS

In [69]:
def calculate_framingham_score(df, age_col, sex_col, ldl_col, trigly_col, hdl_col, systolic_bp_col, bp_treated_col, smoking_col):
    # Function to calculate the individual score for each row
    def calculate_score(row):
        age = row[age_col]
        sex = row[sex_col]  # 1 for male, 0 for female
        hdl = row[hdl_col] * 38.67 # mmol/L to mg/dL
        ldl = row[ldl_col] * 38.67
        trigly = row[trigly_col] * 88.57
        cholesterol = hdl + ldl + (trigly * 0.2)
        systolic_bp = row[systolic_bp_col]
        bp_treated = row[bp_treated_col]
        smoking = row[smoking_col]
        if smoking > 1:
            smoking = 1
        elif smoking < 0:
            smoking = 0
            
        if sex == 0:
            # Age points
            if age < 35: age_points = -7
            elif age < 40: age_points = -3
            elif age < 45: age_points = 0
            elif age < 50: age_points = 3
            elif age < 55: age_points = 6
            elif age < 60: age_points = 8
            elif age < 65: age_points = 10
            elif age < 70: age_points = 12
            elif age < 75: age_points = 14
            else: age_points = 16

            # Cholesterol points
            cholesterol_points = 0
            if age < 40:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [4, 8, 11, 13]
            elif age < 50:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [3, 6, 8, 10]
            elif age < 60:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [2, 4, 5, 7]
            elif age < 70:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [1, 2, 3, 4]
            else:
                cholesterol_ranges = [160, 200, 240]
                cholesterol_scores = [1, 1, 2]

            for i, value in enumerate(cholesterol_ranges):
                if cholesterol >= value:
                    cholesterol_points = cholesterol_scores[i]

            # Smoking points
            smoking_points = 0
            if smoking == 1:
                if age < 40: smoking_points = 9
                elif age < 50: smoking_points = 7
                elif age < 60: smoking_points = 4
                elif age < 70: smoking_points = 2
                else: smoking_points = 1

            # HDL points
            if hdl >= 60: hdl_points = -1
            elif hdl >= 50: hdl_points = 0
            elif hdl >= 40: hdl_points = 1
            else: hdl_points = 2

            # Systolic BP points
            if systolic_bp < 120: bp_points = 0
            elif systolic_bp < 130: bp_points = 1
            elif systolic_bp < 140: bp_points = 2
            elif systolic_bp < 160: bp_points = 3
            else: bp_points = 4

            if bp_treated == 1:
                bp_points += 2

            # Total points
            total_points = age_points + cholesterol_points + smoking_points + hdl_points + bp_points

            # 10-year risk
            risk_categories = [9, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
            risk_percentages = [1, 2, 3, 4, 5, 6, 8, 11, 14, 17, 22, 27]
            risk = 0
            for i, value in enumerate(risk_categories):
                if total_points >= value:
                    risk = risk_percentages[i]
            if total_points >= 25:
                risk = 30

            return risk
        
        else:
            # Age points
            if age < 35: age_points = -9
            elif age < 40: age_points = -4
            elif age < 45: age_points = 0
            elif age < 50: age_points = 3
            elif age < 55: age_points = 6
            elif age < 60: age_points = 8
            elif age < 65: age_points = 10
            elif age < 70: age_points = 11
            elif age < 75: age_points = 12
            else: age_points = 13

            # Cholesterol points
            cholesterol_points = 0
            if age < 40:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [4, 7, 9, 11]
            elif age < 50:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [3, 5, 6, 8]
            elif age < 60:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [2, 3, 4, 5]
            elif age < 70:
                cholesterol_ranges = [160, 200, 240, 280]
                cholesterol_scores = [1, 1, 2, 3]
            else:
                cholesterol_ranges = [240, 280]
                cholesterol_scores = [1, 1]

            for i, value in enumerate(cholesterol_ranges):
                if cholesterol >= value:
                    cholesterol_points = cholesterol_scores[i]

            # Smoking points
            smoking_points = 0
            if smoking == 1:
                if age < 40: smoking_points = 8
                elif age < 50: smoking_points = 5
                elif age < 60: smoking_points = 3
                else: smoking_points = 1

            # HDL points
            if hdl >= 60: hdl_points = -1
            elif hdl >= 50: hdl_points = 0
            elif hdl >= 40: hdl_points = 1
            else: hdl_points = 2

            # Systolic BP points
            if systolic_bp < 130: bp_points = 0
            elif systolic_bp < 140: bp_points = 1
            elif systolic_bp < 160: bp_points = 1
            elif systolic_bp < 160: bp_points = 1
            else: bp_points = 2

            if bp_treated == 1:
                bp_points += 1

            # Total points
            total_points = age_points + cholesterol_points + smoking_points + hdl_points + bp_points

            # 10-year risk
            risk_categories = [1, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
            risk_percentages = [1, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 25]
            risk = 0
            for i, value in enumerate(risk_categories):
                if total_points >= value:
                    risk = risk_percentages[i]
            if total_points >= 17:
                risk = 30

            return risk
    # Applying the calculation to each row
    df['FRS'] = df.apply(calculate_score, axis=1)
    return df


In [70]:
data = calculate_framingham_score(data, '21022-0.0', '31-0.0', 'LDL_direct', 'Triglycerides', 'HDL', '4080-0.0', 'BP_treated', '21000-0.0')

#### SAFE

In [72]:
import numpy as np


# Define a function to calculate the SAFE score based on the provided formula
def calculate_safe_score(df, age_col, bmi_col, diabetes_col, ast_col, alt_col, total_protein_col, albumin_col, platelets_col):
    """
    Calculate the SAFE (Steatosis-Associated Fibrosis Estimator) score.
   
    Parameters:
    age (int): Age of the patient in years.
    bmi (float): Body Mass Index of the patient.
    diabetes (bool): True if the patient has diabetes, False otherwise.
    ast (float): Aspartate aminotransferase level in U/L.
    alt (float): Alanine aminotransferase level in U/L.
    globulins (float): Globulin level in g/dL.
    platelets (float): Platelet count in 10^9/L.
   
    Returns:
    float: The SAFE score.
    """

    def safe(row):
        age = row[age_col]
        bmi = row[bmi_col]
        diabetes = row[diabetes_col]
        ast = row[ast_col]
        alt = row[alt_col]
        globulins = (row[total_protein_col]/10.0) - (row[albumin_col]/10.0)
        platelets = row[platelets_col]

        # Cap BMI at 40 as per the formula        
        bmi = min(bmi, 40)

        # Convert diabetes status to 0 or 1
        diabetes_score = 1 if diabetes else 0

        # Calculate the SAFE score using the provided formula
        safe_score = (2.97 * age) \
                     + (5.99 * bmi) \
                     + (62.85 * diabetes_score) \
                     + (154.85 * np.log(ast)) \
                     - (58.23 * np.log(alt)) \
                     + (195.48 * np.log(globulins)) \
                     - (141.61 * np.log(platelets)) \
                     - 75

        return safe_score

    df['SAFE'] = df.apply(safe, axis=1)
    return df

In [73]:
data = calculate_safe_score(data, '21022-0.0', '21001-0.0_x', 'DM', '30620-0.0', '30650-0.0', 'Total_protein', 'Albumin', '30080-0.0')

In [75]:
data = data.dropna(subset=['SAFE','FIB-4','FRS', 'ASCVD_Risk'])

In [60]:
len(data)

336250

In [76]:
len(data)

336250

## Outcome

In [77]:
import pandas as pd

In [78]:
death_cause = pd.read_csv("./NAFLD-CVD/data/UK biobank_Variables Category 별 나누기/5_Health_outcome.csv"
                         ,usecols=['eid','40001-0.0', '40002-0.1', '40002-0.2', '40002-0.3','40002-0.4', '40002-0.5', '40002-0.6'
                                  , '40002-0.7', '40002-0.8', '40002-0.9','40002-0.10', '40002-0.11', '40002-0.12',
                                   '40002-0.13', '40002-0.14'
                                  ]
                         )

/var/folders/9c/cy0gwyx13172k5yh7sbsrw9r0000gn/T/ipykernel_77054/3620478179.py:1: DtypeWarning: Columns (2847,2849,2850,2851,2852,2853,2854,2855,2856,2857,2858,2859,2860,2861,2862) have mixed types. Specify dtype option on import or set low_memory=False.
  death_cause = pd.read_csv("/Users/charmbong/Documents/NAFLD-CVD/data/UK biobank_Variables Category 별 나누기/5_Health_outcome.csv"


In [79]:
# test = pd.read_csv("./NAFLD-CVD/data/UK biobank_Variables Category 별 나누기/5_Health_outcome.csv", nrows=1)

# secondary --> 14개까지 있음

In [80]:
death_cause

,eid,40001-0.0,40002-0.1,40002-0.2,40002-0.3,40002-0.4,40002-0.5,40002-0.6,40002-0.7,40002-0.8,40002-0.9,40002-0.10,40002-0.11,40002-0.12,40002-0.13,40002-0.14
0,1000012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1000029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1000031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1000047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1000050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502391,6024055,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502392,6024063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502393,6024078,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
502394,6024080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [81]:
death_cause.columns = ['eid', 'icd10', 'sec_1', 'sec_2', 'sec_3',
                      'sec_4', 'sec_5', 'sec_6','sec_7', 'sec_8', 'sec_9',
                      'sec_10', 'sec_11', 'sec_12', 'sec_13','sec_14']

In [82]:
target_columns = ['eid', 'icd10', 'sec_1', 'sec_2', 'sec_3',
                      'sec_4', 'sec_5', 'sec_6','sec_7', 'sec_8', 'sec_9',
                      'sec_10', 'sec_11', 'sec_12', 'sec_13','sec_14']  # 대상이 되는 여러 열
target_codes = ('I21', 'I22', 'I23', 'I24', 'I25', 'I50', 'I60', 'I61', 'I63', 'I64')

# 여러 열에서 조건을 만족하면 1, 아니면 0
death_cause['cardiac_death'] = death_cause[target_columns].applymap(
    lambda x: int(str(x).startswith(target_codes)) if pd.notnull(x) else 0
).max(axis=1)  # 행(row) 기준으로 하나라도 1이면 1로 설정


/var/folders/9c/cy0gwyx13172k5yh7sbsrw9r0000gn/T/ipykernel_77054/315524935.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  death_cause['cardiac_death'] = death_cause[target_columns].applymap(


In [96]:
death_cause.cardiac_death.value_counts()

0    492932
1      9464
Name: cardiac_death, dtype: int64

In [83]:
death_cause.cardiac_death.value_counts()

cardiac_death
0    492932
1      9464
Name: count, dtype: int64

In [84]:
#death_cause.columns = ['eid','icd10', 'secondary', 'sec_2','sec_3', 'cardiac_death']
death_cause = death_cause.query('cardiac_death == 1').eid.tolist()
data['cardiac_death'] = data['eid'].isin(death_cause).astype(int)

In [85]:
data.cardiac_death.value_counts()

cardiac_death
0    331458
1      4792
Name: count, dtype: int64

In [86]:
outcomes = pd.read_csv("./NAFLD-CVD/data/algorithm_defined.csv")
target_codes = ('I21', 'I22', 'I23', 'I24', 'I25', 'I50', 'I60', 'I61', 'I63', 'I64')
target_columns = ['41270-0.0', '41202-0.0', '41204-0.0']  # 대상이 되는 여러 열

# 여러 열에서 조건을 만족하면 1, 아니면 0
outcomes['cardiac_dx'] = outcomes[target_columns].applymap(
    lambda x: int(str(x).startswith(target_codes)) if pd.notnull(x) else 0
).max(axis=1)  # 행(row) 기준으로 하나라도 1이면 1로 설정
outcome = outcomes[['eid','42000-0.0', '42002-0.0','42004-0.0','42006-0.0','42008-0.0', '42010-0.0', '42012-0.0']]
outcome.columns = ['eid','MI', 'STEMI', 'NSTEMI', 'STROKE', 'ISCHAEMIC', 'INTRACEREBRAL_HEMO', 'SUBARACHNOID_HEMO']

In [87]:
outcome = outcome.query('MI == MI or STEMI == STEMI or NSTEMI == NSTEMI or STROKE == STROKE or ISCHAEMIC == ISCHAEMIC or INTRACEREBRAL_HEMO == INTRACEREBRAL_HEMO or SUBARACHNOID_HEMO == SUBARACHNOID_HEMO')

In [88]:
outcome.columns

Index(['eid', 'MI', 'STEMI', 'NSTEMI', 'STROKE', 'ISCHAEMIC',
       'INTRACEREBRAL_HEMO', 'SUBARACHNOID_HEMO'],
      dtype='object')

In [89]:
outcome = outcome.merge(data[['eid', '53-0.0']], on=['eid'])
outcome.columns = ['eid', 'MI', 'STEMI', 'NSTEMI', 'STROKE', 'ISCHAEMIC',
       'INTRACEREBRAL_HEMO', 'SUBARACHNOID_HEMO', 'baseline']
outcome = outcome.query('MI > baseline or STEMI > baseline or NSTEMI > baseline or STROKE > baseline or ISCHAEMIC > baseline or INTRACEREBRAL_HEMO > baseline or SUBARACHNOID_HEMO > baseline')

In [90]:
outcome['MI_bi'] = 1
outcome.loc[outcome['MI'].isna(), 'MI_bi'] = 0

outcome['STROKE_bi'] = 1
outcome.loc[outcome['STROKE'].isna(), 'STROKE_bi'] = 0

outcome['STEMI_bi'] = 1
outcome.loc[outcome['STEMI'].isna(), 'STEMI_bi'] = 0

outcome['NSTEMI_bi'] = 1
outcome.loc[outcome['NSTEMI'].isna(), 'NSTEMI_bi'] = 0

outcome['ISCHAEMIC_bi'] = 1
outcome.loc[outcome['ISCHAEMIC'].isna(), 'ISCHAEMIC_bi'] = 0

outcome['INTRACEREBRAL_HEMO_bi'] = 1
outcome.loc[outcome['INTRACEREBRAL_HEMO'].isna(), 'INTRACEREBRAL_HEMO_bi'] = 0

outcome['SUBARACHNOID_HEMO_bi'] = 1
outcome.loc[outcome['SUBARACHNOID_HEMO'].isna(), 'SUBARACHNOID_HEMO_bi'] = 0


data = data.merge(outcome[['eid','cardiac_dx', 'MI_bi', 'STROKE_bi', 'STEMI_bi', 'NSTEMI_bi', 
                           'ISCHAEMIC_bi', 'INTRACEREBRAL_HEMO_bi', 'SUBARACHNOID_HEMO_bi']], on=['eid'], how='left')

In [91]:
data['outcome'] = (
    data['cardiac_death'].fillna(0).astype(int) |
    data['cardiac_dx'].fillna(0).astype(int) | 
    data['MI_bi'].fillna(0).astype(int) | 
    data['STROKE_bi'].fillna(0).astype(int) |
    data['STEMI_bi'].fillna(0).astype(int) |
    data['NSTEMI_bi'].fillna(0).astype(int) |
    data['ISCHAEMIC_bi'].fillna(0).astype(int) |
    data['INTRACEREBRAL_HEMO_bi'].fillna(0).astype(int) |
    data['SUBARACHNOID_HEMO_bi'].fillna(0).astype(int)
).astype(int)

### Dyslipidemia

In [94]:
medication = pd.read_csv('./NAFLD-CVD/data/data5.csv')

/var/folders/9c/cy0gwyx13172k5yh7sbsrw9r0000gn/T/ipykernel_77054/2266877816.py:1: DtypeWarning: Columns (221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,243,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276) have mixed types. Specify dtype option on import or set low_memory=False.
  medication = pd.read_csv('/Users/charmbong/Documents/NAFLD-CVD/data/data5.csv')


In [95]:
## Check Dyslipidemia
medication['6153-0.0'].value_counts()
dyslipidemia_man = medication[medication['6177-0.0'] == 1.0]
dyslipidemia_woman = medication[medication['6153-0.0'] == 1.0]
dyslipid_l_m = dyslipidemia_man.eid.tolist()
dyslipid_l_wm = dyslipidemia_man.eid.tolist()
data['Dyslipidemia'] = data['eid'].apply(lambda x: 1 if x in dyslipid_l_m or x in dyslipid_l_wm else 0)

In [96]:
del medication

In [97]:
data.to_csv("./NAFLD-CVD/data/ukb_final_data(260304)_addDeath.csv", index=False)